# 02 · Integration and clustering — GSE216146

Highly variable genes, PCA, batch correction with **Harmony** (one batch per `biosample`), neighbor graph,
UMAP and Leiden clustering at several resolutions.

**Input:** `results/01_qc.h5ad` · **Output:** `results/02_integrated.h5ad` with
`obsm["X_pca"]`, `obsm["X_pca_harmony"]`, `obsm["X_umap"]` (Harmony), `obsm["X_umap_uncorrected"]`,
and `obs["leiden_res_<r>"]` for each resolution in `config.yaml`.

Things to check here:
- Samples and batches (`D20`, `D21`) should mix after Harmony, while control and chemotherapy cells of the
  same type should overlap.
- Batch D20 has more low-complexity, high-mt cells (see `01_qc`). Look for clusters made mostly of D20 cells
  with low `n_genes_by_counts` or high `pct_counts_mt`: they are candidates for low-quality cells, not cell types.

In [ ]:
import gc
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["integration"]
seed = p["random_state"]
scu.setup_plotting()
print("scanpy", version("scanpy"), "| harmonypy", version("harmonypy"))
p

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "01_qc.h5ad")
adata

## 1. Highly variable genes and PCA

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=p["n_top_genes"], batch_key=p["hvg_batch_key"])
print("HVGs:", int(adata.var["highly_variable"].sum()))

fig = sc.pl.highly_variable_genes(adata, show=False)
scu.save_fig(plt.gcf(), cfg, "02_hvg")
plt.show()

In [ ]:
sc.pp.pca(adata, n_comps=p["n_pcs"], mask_var="highly_variable", random_state=seed)

sc.pl.pca_variance_ratio(adata, n_pcs=p["n_pcs"], log=True, show=False)
scu.save_fig(plt.gcf(), cfg, "02_pca_variance_ratio")
plt.show()

## 2. Embedding without batch correction (reference)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=p["n_neighbors"], use_rep="X_pca", random_state=seed)
sc.tl.umap(adata, random_state=seed)
adata.obsm["X_umap_uncorrected"] = adata.obsm["X_umap"].copy()

fig = sc.pl.embedding(adata, basis="umap_uncorrected", color=["biosample", "batch", "condition"],
                      ncols=3, wspace=0.35, size=8, return_fig=True)
scu.save_fig(fig, cfg, "02_umap_uncorrected")
plt.show()

## 3. Harmony

In [ ]:
# harmonypy is called directly (see scu.run_harmony): scanpy's wrapper mis-orients harmonypy >= 2 output
scu.run_harmony(adata, key=p["harmony_key"], basis="X_pca", adjusted_basis="X_pca_harmony", random_state=seed)

sc.pp.neighbors(adata, n_neighbors=p["n_neighbors"], use_rep="X_pca_harmony", random_state=seed)
sc.tl.umap(adata, random_state=seed)

fig = sc.pl.umap(adata, color=["biosample", "batch", "condition"], ncols=3, wspace=0.35, size=8, return_fig=True)
scu.save_fig(fig, cfg, "02_umap_harmony")
plt.show()

In [ ]:
# Author labels (from the GEO object) as a sanity check of the corrected embedding
fig = sc.pl.umap(adata, color=["MajorCellType"], size=8, return_fig=True)
scu.save_fig(fig, cfg, "02_umap_author_celltypes")
plt.show()

## 4. Leiden clustering

In [ ]:
res_keys = []
for res in p["leiden_resolutions"]:
    key = f"leiden_res_{res:.2f}"
    sc.tl.leiden(adata, resolution=res, key_added=key, flavor="igraph",
                 n_iterations=2, directed=False, random_state=seed)
    res_keys.append(key)
    print(key, adata.obs[key].nunique(), "clusters")

fig = sc.pl.umap(adata, color=res_keys, ncols=3, wspace=0.35, size=8, legend_loc="on data",
                 legend_fontsize=8, return_fig=True)
scu.save_fig(fig, cfg, "02_umap_leiden")
plt.show()

## 5. Cluster quality and batch mixing

Main resolution for the checks below: the middle value in `config.yaml`. A cluster dominated by one sample or batch,
or with low genes / high mt, needs a closer look before annotation.

In [ ]:
main_key = res_keys[len(res_keys) // 2]

fig = sc.pl.umap(adata, color=["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
                 ncols=4, wspace=0.3, size=8, cmap="viridis", return_fig=True)
scu.save_fig(fig, cfg, "02_umap_qc_metrics")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["n_genes_by_counts", "pct_counts_mt"]):
    sc.pl.violin(adata, col, groupby=main_key, ax=ax, show=False, size=0.5, stripplot=False)
    ax.set_title(f"{col} by {main_key}")
fig.tight_layout()
scu.save_fig(fig, cfg, "02_cluster_qc_violin")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
scu.plot_fractions(adata.obs, main_key, "biosample", ax=axes[0])
scu.plot_fractions(adata.obs, main_key, "condition", ax=axes[1])
fig.tight_layout()
scu.save_fig(fig, cfg, "02_cluster_composition")
plt.show()

In [ ]:
# Per-cluster summary: size, batch/condition mix, QC medians, majority author label
obs = adata.obs
cluster_summary = pd.concat([
    obs.groupby(main_key, observed=True).size().rename("cells"),
    scu.fraction_table(obs, main_key, "batch").add_prefix("frac_"),
    scu.fraction_table(obs, main_key, "condition").add_prefix("frac_"),
    obs.groupby(main_key, observed=True)[["n_genes_by_counts", "pct_counts_mt"]].median().add_prefix("median_"),
    obs.groupby(main_key, observed=True)["MajorCellType"].agg(lambda s: s.value_counts().index[0]).rename("author_majority"),
    obs.groupby(main_key, observed=True)["MajorCellType"].agg(lambda s: s.value_counts(normalize=True).iloc[0]).rename("author_majority_frac"),
], axis=1).round(3)
cluster_summary.to_csv(cfg["paths"]["tables_dir"] / f"02_cluster_summary_{main_key}.tsv", sep="\t")
cluster_summary

## 6. Save

In [ ]:
out_h5ad = cfg["paths"]["results_dir"] / "02_integrated.h5ad"
adata.write_h5ad(out_h5ad, compression="gzip")
print("wrote", out_h5ad)
adata

In [ ]:
del adata
gc.collect()